# 03 Agent01 Download Contract

Este notebook fija el contrato esperado de `Agent01` y lo compara con la ejecución real del run auditado.  
**“¿qué debía hacer exactamente Agent01 y qué estado operativo dejó?”**   
**“¿Qué se esperaba exactamente de Agent01 y qué hizo realmente en este run?”** 
sirve para auditar a Agent01 como componente, no para mirar coverage ni validar calidad semántica.

Sirve para separar y revisar:

- cuál era el input maestro de descarga
- cuál fue el input recovery actual
- qué estado dejó Agent01
- qué artefactos produce
- qué responsabilidades sí tiene
- qué responsabilidades no debería tener

Deja explícito que Agent01 debe:

- consumir un tasks.csv
- escribir parquets en D:\quotes
- mantener history/current/state/live_status
- hacer integridad física mínima
- no decidir universo
- no construir tasks
- no redefinir lifecycle ni calendario

Distingue entre:

- tasks_quotes_prod.csv
    - contrato maestro
- tasks_quotes_prod_missing_only_final_v2.csv
    - contrato recovery actual

Eso es importante porque ahora mismo tu descarga en curso no está corriendo contra el contrato maestro, sino contra el
de recovery.

**El estado real dejado por Agent01**

Lee:

- download_events_current.csv
- download_events_history.csv
- download_state.json
- download_live_status.json

para ver:

- cuántas tareas vio
- cuántas procesó
- qué estados cerró
- qué archivo de entrada está usando realmente

**Qué no hace este notebook**

No sirve para:

- reconstruir el universo
- decidir si las tasks están bien construidas
- validar si los parquets son “buenos” para research
- medir cobertura final
- decidir go/no-go del dataset

Eso corresponde más a:

- 01_universe_lineage_quotes.ipynb
- 02_task_builder_contract_quotes.ipynb
- y después a los notebooks de Agent02 y Agent03

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps")
RUN_ID = "20260313_quotes_prod_full_12133_clean"
RUN_DIR = PROJECT_ROOT / "runs" / "polygon_realtime_audit" / RUN_ID
INPUTS_DIR = RUN_DIR / "inputs"
QUOTES_ROOT = Path(r"D:\quotes")

TASKS_INPUT_MASTER = INPUTS_DIR / "tasks_quotes_prod.csv"
TASKS_INPUT_RECOVERY = INPUTS_DIR / "tasks_quotes_prod_missing_only_final_v2.csv"
DOWNLOAD_EVENTS_CURRENT = RUN_DIR / "download_events_current.csv"
DOWNLOAD_EVENTS_HISTORY = RUN_DIR / "download_events_history.csv"
DOWNLOAD_STATE_JSON = RUN_DIR / "download_state.json"
DOWNLOAD_LIVE_STATUS_JSON = RUN_DIR / "download_live_status.json"

def file_snapshot(path: Path) -> dict:
    return {
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else None,
    }

snapshots = pd.DataFrame([
    {"artifact": "tasks_master", **file_snapshot(TASKS_INPUT_MASTER)},
    {"artifact": "tasks_recovery", **file_snapshot(TASKS_INPUT_RECOVERY)},
    {"artifact": "download_events_current", **file_snapshot(DOWNLOAD_EVENTS_CURRENT)},
    {"artifact": "download_events_history", **file_snapshot(DOWNLOAD_EVENTS_HISTORY)},
    {"artifact": "download_state_json", **file_snapshot(DOWNLOAD_STATE_JSON)},
    {"artifact": "download_live_status_json", **file_snapshot(DOWNLOAD_LIVE_STATUS_JSON)},
])

state = json.loads(DOWNLOAD_STATE_JSON.read_text(encoding="utf-8")) if DOWNLOAD_STATE_JSON.exists() else {}
live = json.loads(DOWNLOAD_LIVE_STATUS_JSON.read_text(encoding="utf-8")) if DOWNLOAD_LIVE_STATUS_JSON.exists() else {}

agent01_contract = pd.DataFrame([
    {"domain": "input", "rule": "consume solo un tasks.csv explícito"},
    {"domain": "universe", "rule": "no decide universo ni crea tasks"},
    {"domain": "write_path", "rule": r"escribe quotes en D:\\quotes\\<ticker>\\year=YYYY\\month=MM\\day=DD\
\quotes.parquet"},
    {"domain": "state", "rule": "mantiene history/current/state/live_status"},
    {"domain": "integrity", "rule": "usa tmp + replace atómico + relectura del parquet"},
    {"domain": "resume", "rule": "solo debe saltarse tareas cerradas según política explícita"},
    {"domain": "forbidden", "rule": "no redefine lifecycle, calendar ni target population"},
])

print("RUN_DIR:", RUN_DIR)
print("QUOTES_ROOT:", QUOTES_ROOT)
print("\nArtifacts snapshot:")
display(snapshots)

print("\nDOWNLOAD_STATE_JSON:")
print(json.dumps(state, indent=2, ensure_ascii=False)[:3000] if state else "{}")

print("\nDOWNLOAD_LIVE_STATUS_JSON:")
print(json.dumps(live, indent=2, ensure_ascii=False)[:3000] if live else "{}")

print("\nAgent01 contract:")
display(agent01_contract)

RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean
QUOTES_ROOT: D:\quotes

Artifacts snapshot:


,artifact,path,exists,size_mb
0,tasks_master,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,49.21
1,tasks_recovery,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,14.54
2,download_events_current,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,149.62
3,download_events_history,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,506.51
4,download_state_json,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,0.00
5,download_live_status_json,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,0.00



DOWNLOAD_STATE_JSON:
{
  "run_id": "20260313_quotes_prod_full_12133_clean",
  "updated_utc": "2026-03-19T18:01:47.227866+00:00",
  "tasks_total": 907151,
  "tasks_current_rows": 749581,
  "status_counts_current": {
    "DOWNLOADED_OK": 544701,
    "DOWNLOADED_EMPTY": 204880
  },
  "retry_pending": 0,
  "output_root": "D:\\quotes",
  "csv_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260313_quotes_prod_full_12133_clean\\inputs\\tasks_quotes_prod_missing_only_final_v2.csv"
}

DOWNLOAD_LIVE_STATUS_JSON:
{
  "updated_utc": "2026-03-19T18:01:47.282200+00:00",
  "run_id": "20260313_quotes_prod_full_12133_clean",
  "output_root": "D:\\quotes",
  "tasks_total": 907151,
  "done_ok": 749581,
  "done_bad": 0,
  "pending": 157570,
  "status_counts_current": {
    "DOWNLOADED_OK": 544701,
    "DOWNLOADED_EMPTY": 204880
  },
  "concurrent": 24,
  "max_pages": 0,
  "session_start_local": "04:00:00",
  "session_end_local": "20:00:00"
}

Agent01 contract:


,domain,rule
0,input,consume solo un tasks.csv explícito
1,universe,no decide universo ni crea tasks
2,write_path,escribe quotes en D:\\quotes\\<ticker>\\year=Y...
3,state,mantiene history/current/state/live_status
4,integrity,usa tmp + replace atómico + relectura del parquet
5,resume,solo debe saltarse tareas cerradas según polít...
6,forbidden,"no redefine lifecycle, calendar ni target popu..."


In [4]:
LOAD_HISTORY = False
LOAD_CURRENT = True
LOAD_TASKS_MASTER = True
LOAD_TASKS_RECOVERY = True

tasks_master = (
    pd.read_csv(TASKS_INPUT_MASTER, usecols=["ticker", "date"])
    if LOAD_TASKS_MASTER and TASKS_INPUT_MASTER.exists()
    else pd.DataFrame()
)

tasks_recovery = (
    pd.read_csv(TASKS_INPUT_RECOVERY, usecols=["ticker", "date"])
    if LOAD_TASKS_RECOVERY and TASKS_INPUT_RECOVERY.exists()
    else pd.DataFrame()
)

current = (
    pd.read_csv(
        DOWNLOAD_EVENTS_CURRENT,
        usecols=["task_key", "ticker", "date", "status", "rows", "error", "file", "processed_at_utc", "run_id"],
    )
    if LOAD_CURRENT and DOWNLOAD_EVENTS_CURRENT.exists()
    else pd.DataFrame()
)

history = (
    pd.read_csv(
        DOWNLOAD_EVENTS_HISTORY,
        usecols=["task_key", "ticker", "date", "status", "rows", "error", "file", "processed_at_utc", "run_id"],
        nrows=200000,
    )
    if LOAD_HISTORY and DOWNLOAD_EVENTS_HISTORY.exists()
    else pd.DataFrame()
)

summary = {
    "tasks_master_rows": int(len(tasks_master)) if not tasks_master.empty else 0,
    "tasks_master_unique_tickers": int(tasks_master["ticker"].nunique()) if "ticker" in tasks_master.columns else 0,
    "tasks_recovery_rows": int(len(tasks_recovery)) if not tasks_recovery.empty else 0,
    "tasks_recovery_unique_tickers": int(tasks_recovery["ticker"].nunique()) if "ticker" in tasks_recovery.columns
else 0,
    "current_rows": int(len(current)) if not current.empty else 0,
    "current_status_counts": current["status"].value_counts(dropna=False).to_dict() if "status" in current.columns
else {},
    "history_rows_loaded": int(len(history)) if not history.empty else 0,
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

if not current.empty:
    display(current.head(10))

if not tasks_recovery.empty:
    display(tasks_recovery.head(10))

{
  "tasks_master_rows": 3067682,
  "tasks_master_unique_tickers": 1961,
  "tasks_recovery_rows": 907151,
  "tasks_recovery_unique_tickers": 1582,
  "current_rows": 750881,
  "current_status_counts": {
    "DOWNLOADED_OK": 545948,
    "DOWNLOADED_EMPTY": 204933
  },
  "history_rows_loaded": 0
}


,task_key,ticker,date,file,status,rows,error,processed_at_utc,run_id
0,AABA|2017-06-16,AABA,2017-06-16,D:\quotes\AABA\year=2017\month=06\day=16\quote...,DOWNLOADED_EMPTY,0.0,polygon_empty_results,2026-03-12 23:34:00.524302+00:00,20260313_quotes_prod_full_12133_clean
1,AABA|2017-06-19,AABA,2017-06-19,D:\quotes\AABA\year=2017\month=06\day=19\quote...,DOWNLOADED_OK,156436.0,NaN,2026-03-12 23:37:33.439816+00:00,20260313_quotes_prod_full_12133_clean
2,AABA|2017-06-20,AABA,2017-06-20,D:\quotes\AABA\year=2017\month=06\day=20\quote...,DOWNLOADED_OK,123512.0,NaN,2026-03-12 23:33:25.668457+00:00,20260313_quotes_prod_full_12133_clean
3,AABA|2017-06-21,AABA,2017-06-21,D:\quotes\AABA\year=2017\month=06\day=21\quote...,DOWNLOADED_OK,155986.0,NaN,2026-03-12 23:35:19.632236+00:00,20260313_quotes_prod_full_12133_clean
4,AABA|2017-06-22,AABA,2017-06-22,D:\quotes\AABA\year=2017\month=06\day=22\quote...,DOWNLOADED_OK,166944.0,NaN,2026-03-12 23:37:50.964307+00:00,20260313_quotes_prod_full_12133_clean
5,AABA|2017-06-23,AABA,2017-06-23,D:\quotes\AABA\year=2017\month=06\day=23\quote...,DOWNLOADED_OK,150906.0,NaN,2026-03-12 23:38:08.049764+00:00,20260313_quotes_prod_full_12133_clean
6,AABA|2017-06-26,AABA,2017-06-26,D:\quotes\AABA\year=2017\month=06\day=26\quote...,DOWNLOADED_OK,307979.0,NaN,2026-03-12 23:32:02.198130+00:00,20260313_quotes_prod_full_12133_clean
7,AABA|2017-06-27,AABA,2017-06-27,D:\quotes\AABA\year=2017\month=06\day=27\quote...,DOWNLOADED_OK,227019.0,NaN,2026-03-12 23:31:09.180953+00:00,20260313_quotes_prod_full_12133_clean
8,AABA|2017-06-28,AABA,2017-06-28,D:\quotes\AABA\year=2017\month=06\day=28\quote...,DOWNLOADED_OK,211289.0,NaN,2026-03-12 23:34:35.768629+00:00,20260313_quotes_prod_full_12133_clean
9,AABA|2017-06-29,AABA,2017-06-29,D:\quotes\AABA\year=2017\month=06\day=29\quote...,DOWNLOADED_OK,269659.0,NaN,2026-03-12 23:40:48.696926+00:00,20260313_quotes_prod_full_12133_clean


,ticker,date
0,AACI,2026-03-13
1,AAT,2026-03-13
2,ABAT,2026-03-13
3,ABCL,2026-03-13
4,ABLV,2026-03-13
5,ABOS,2026-03-13
6,ABSI,2026-03-13
7,ABVC,2026-03-13
8,ABVE,2026-03-13
9,ACCL,2026-03-13


**Esta te da un resumen rápido de download_events_current.csv sin cargar history.**

In [2]:
import json
import pandas as pd

CURRENT_SUMMARY_NROWS = None  # deja None para leer current completo; suele ser asumible comparado con history

current_summary = (
    pd.read_csv(
        DOWNLOAD_EVENTS_CURRENT,
        usecols=["task_key", "ticker", "date", "status", "rows", "error", "file", "processed_at_utc", "run_id"],
        nrows=CURRENT_SUMMARY_NROWS,
    )
    if DOWNLOAD_EVENTS_CURRENT.exists()
    else pd.DataFrame()
)

if current_summary.empty:
    print("download_events_current.csv no existe o está vacío")
else:
    summary = {
        "rows_loaded": int(len(current_summary)),
        "unique_task_keys": int(current_summary["task_key"].nunique()) if "task_key" in current_summary.columns else
None,
        "unique_tickers": int(current_summary["ticker"].nunique()) if "ticker" in current_summary.columns else None,
        "date_min": str(pd.to_datetime(current_summary["date"], errors="coerce").min().date()) if "date" in
current_summary.columns else None,
        "date_max": str(pd.to_datetime(current_summary["date"], errors="coerce").max().date()) if "date" in
current_summary.columns else None,
        "status_counts": current_summary["status"].value_counts(dropna=False).to_dict() if "status" in
current_summary.columns else {},
        "top_errors": current_summary["error"].fillna("<<NA>>").value_counts(dropna=False).head(15).to_dict() if
"error" in current_summary.columns else {},
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))

    print("\nStatus sample:")
    display(current_summary.head(10))

    print("\nRows by status:")
    display(
        current_summary["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )

    print("\nTop errors:")
    display(
        current_summary["error"]
        .fillna("<<NA>>")
        .value_counts(dropna=False)
        .head(15)
        .rename_axis("error")
        .reset_index(name="count")
    )

{
  "rows_loaded": 750681,
  "unique_task_keys": 750681,
  "unique_tickers": 1338,
  "date_min": "1996-05-30",
  "date_max": "2026-03-13",
  "status_counts": {
    "DOWNLOADED_OK": 545753,
    "DOWNLOADED_EMPTY": 204928
  },
  "top_errors": {
    "<<NA>>": 538662,
    "polygon_empty_results": 204928,
    "resume_existing_file": 7091
  }
}

Status sample:


,task_key,ticker,date,file,status,rows,error,processed_at_utc,run_id
0,AABA|2017-06-16,AABA,2017-06-16,D:\quotes\AABA\year=2017\month=06\day=16\quote...,DOWNLOADED_EMPTY,0.0,polygon_empty_results,2026-03-12 23:34:00.524302+00:00,20260313_quotes_prod_full_12133_clean
1,AABA|2017-06-19,AABA,2017-06-19,D:\quotes\AABA\year=2017\month=06\day=19\quote...,DOWNLOADED_OK,156436.0,NaN,2026-03-12 23:37:33.439816+00:00,20260313_quotes_prod_full_12133_clean
2,AABA|2017-06-20,AABA,2017-06-20,D:\quotes\AABA\year=2017\month=06\day=20\quote...,DOWNLOADED_OK,123512.0,NaN,2026-03-12 23:33:25.668457+00:00,20260313_quotes_prod_full_12133_clean
3,AABA|2017-06-21,AABA,2017-06-21,D:\quotes\AABA\year=2017\month=06\day=21\quote...,DOWNLOADED_OK,155986.0,NaN,2026-03-12 23:35:19.632236+00:00,20260313_quotes_prod_full_12133_clean
4,AABA|2017-06-22,AABA,2017-06-22,D:\quotes\AABA\year=2017\month=06\day=22\quote...,DOWNLOADED_OK,166944.0,NaN,2026-03-12 23:37:50.964307+00:00,20260313_quotes_prod_full_12133_clean
5,AABA|2017-06-23,AABA,2017-06-23,D:\quotes\AABA\year=2017\month=06\day=23\quote...,DOWNLOADED_OK,150906.0,NaN,2026-03-12 23:38:08.049764+00:00,20260313_quotes_prod_full_12133_clean
6,AABA|2017-06-26,AABA,2017-06-26,D:\quotes\AABA\year=2017\month=06\day=26\quote...,DOWNLOADED_OK,307979.0,NaN,2026-03-12 23:32:02.198130+00:00,20260313_quotes_prod_full_12133_clean
7,AABA|2017-06-27,AABA,2017-06-27,D:\quotes\AABA\year=2017\month=06\day=27\quote...,DOWNLOADED_OK,227019.0,NaN,2026-03-12 23:31:09.180953+00:00,20260313_quotes_prod_full_12133_clean
8,AABA|2017-06-28,AABA,2017-06-28,D:\quotes\AABA\year=2017\month=06\day=28\quote...,DOWNLOADED_OK,211289.0,NaN,2026-03-12 23:34:35.768629+00:00,20260313_quotes_prod_full_12133_clean
9,AABA|2017-06-29,AABA,2017-06-29,D:\quotes\AABA\year=2017\month=06\day=29\quote...,DOWNLOADED_OK,269659.0,NaN,2026-03-12 23:40:48.696926+00:00,20260313_quotes_prod_full_12133_clean



Rows by status:


,status,count
0,DOWNLOADED_OK,545753
1,DOWNLOADED_EMPTY,204928



Top errors:


,error,count
0,<<NA>>,538662
1,polygon_empty_results,204928
2,resume_existing_file,7091


In [5]:
snapshot = {
    "tasks_master_rows": int(len(tasks_master)),
    "tasks_master_unique_tickers": int(tasks_master['ticker'].astype(str).nunique()) if 'ticker' in tasks_master.columns else None,
    "tasks_recovery_rows": int(len(tasks_recovery)),
    "tasks_recovery_unique_tickers": int(tasks_recovery['ticker'].astype(str).nunique()) if 'ticker' in tasks_recovery.columns else None,
    "current_rows": int(len(current)),
    "history_rows": int(len(history)),
    "state": state,
    "live": live,
}
print(json.dumps(snapshot, indent=2, ensure_ascii=False))

{
  "tasks_master_rows": 3067682,
  "tasks_master_unique_tickers": 1961,
  "tasks_recovery_rows": 907151,
  "tasks_recovery_unique_tickers": 1582,
  "current_rows": 750881,
  "history_rows": 0,
  "state": {
    "run_id": "20260313_quotes_prod_full_12133_clean",
    "updated_utc": "2026-03-19T18:01:47.227866+00:00",
    "tasks_total": 907151,
    "tasks_current_rows": 749581,
    "status_counts_current": {
      "DOWNLOADED_OK": 544701,
      "DOWNLOADED_EMPTY": 204880
    },
    "retry_pending": 0,
    "output_root": "D:\\quotes",
    "csv_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260313_quotes_prod_full_12133_clean\\inputs\\tasks_quotes_prod_missing_only_final_v2.csv"
  },
  "live": {
    "updated_utc": "2026-03-19T18:01:47.282200+00:00",
    "run_id": "20260313_quotes_prod_full_12133_clean",
    "output_root": "D:\\quotes",
    "tasks_total": 907151,
    "done_ok": 749581,
    "done_bad": 0,
    "pending": 157570,
    "status_counts_current": {
  

In [6]:
agent01_inputs = pd.DataFrame([
    {
        "contract_kind": "master_contract",
        "input_path": str(TASKS_INPUT_MASTER),
        "exists": TASKS_INPUT_MASTER.exists(),
        "rows": int(len(tasks_master)),
    },
    {
        "contract_kind": "recovery_contract",
        "input_path": str(TASKS_INPUT_RECOVERY),
        "exists": TASKS_INPUT_RECOVERY.exists(),
        "rows": int(len(tasks_recovery)),
    },
    {
        "contract_kind": "agent01_runtime_contract",
        "input_path": str(state.get('csv_path')),
        "exists": Path(state['csv_path']).exists() if state.get('csv_path') else False,
        "rows": int(state.get('tasks_total')) if state.get('tasks_total') is not None else None,
    },
])
agent01_inputs

,contract_kind,input_path,exists,rows
0,master_contract,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,3067682
1,recovery_contract,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,907151
2,agent01_runtime_contract,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,True,907151


In [7]:
if len(current):
    current['task_key_inferred'] = current['ticker'].astype(str) + '|' + current['date'].astype(str)
    current['file_exists_now'] = current['file'].astype(str).map(lambda p: Path(p).exists())
    summary = {
        'status_counts': current['status'].value_counts(dropna=False).to_dict(),
        'files_existing_now': int(current['file_exists_now'].sum()),
        'files_missing_now': int((~current['file_exists_now']).sum()),
        'duplicates_task_key': int(current.duplicated(['task_key_inferred']).sum()),
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    display(current.head(10))
else:
    print('download_events_current.csv no existe o está vacío')

KeyboardInterrupt: 

**Voy a comprobar cuantos files en local hay versus universo esperado**

He creado la lanzadera aquí:

- run_quotes_disk_audit.ps1

Qué hace:

- inventaría D:\quotes y deriva task_key = ticker|date
- compara contra:
    - tasks_quotes_prod.csv
    - tasks_quotes_prod_missing_only_final_v2.csv
    - download_events_current.csv
    - quotes_agent_strict_events_current.csv
- escribe CSVs y un JSON resumen dentro del run_dir

Prueba pequeña con 3 files y progreso en cada file:

```bash
powershell -ExecutionPolicy Bypass -File C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification_v2\cell_code\run_quotes_disk_audit.ps1 -MaxFiles 3 -ProgressEvery 1 -OutputPrefix smoke3
```

Lánzalo así:

```bash
powershell -ExecutionPolicy Bypass -File C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification_v2\cell_code\run_quotes_disk_audit.ps1 -ProgressEvery 10000
```


Salidas principales que te dejará en el run_dir:

- disk_quotes_inventory_taskkey_live_audit.csv
- audit_disk_vs_master_live_audit.csv
- audit_master_missing_on_disk_live_audit.csv
- audit_disk_not_in_master_live_audit.csv
- audit_disk_vs_recovery_live_audit.csv
- audit_recovery_missing_on_disk_live_audit.csv
- audit_current_closed_not_on_disk_live_audit.csv
- audit_disk_not_in_current_live_audit.csv
- audit_disk_vs_agent02_live_audit.csv
- audit_disk_summary_live_audit.json


Y sí: los archivos de referencia directos que usan los notebooks 01-04 ya están. Los verificados como presentes son:

- tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet
- official_lifecycle_compiled.csv
- tickers_quotes_prod.csv
- tasks_quotes_prod.csv
- tasks_quotes_prod_meta.json
- tasks_quotes_prod_missing_only_final_v2.csv
- download_events_current.csv
- download_events_history.csv
- download_state.json
- download_live_status.json
- quotes_agent_strict_events_current.csv

Matiz fino: para saber el universo exacto que debía haber en el run histórico, la referencia final inmediata es
tickers_quotes_prod.csv; el parquet refinado y el lifecycle son las fuentes upstream que explican cómo se llegó ahí.

----

He añadido la segunda lanzadera aquí:

  - build_clean_quotes_run.ps1

  Qué hace:

  - toma el universo refinado y el lifecycle oficial
  - recorta cada ticker a la ventana explícita del run
  - genera:
      - tickers_quotes_prod_v2_clean.csv
      - tasks_quotes_prod_v2_clean.csv
      - tasks_quotes_prod_v2_clean.meta.json
  - te imprime el comando exacto de Agent01 para ese run limpio

  Uso recomendado, con root separado para no mezclar con la descarga actual:

```bash
powershell -ExecutionPolicy Bypass -File C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification_v2\cell_code\build_clean_quotes_run.ps1 -RunId 20260319_quotes_clean_v2_draft -QuotesRoot D:\quotes_clean_v2 -RunDateFrom 2005-01-01 -RunDateTo 2025-12-31
```

  Y ya tienes las dos lanzaderas:

  - Auditoría del disco actual:
      - run_quotes_disk_audit.ps1
  - Preparación del run limpio v2:
      - build_clean_quotes_run.ps1

  Mi recomendación operativa:

  - mientras siga corriendo la descarga actual, usa solo run_quotes_disk_audit.ps1
  - cuando quieras preparar la nueva corrida limpia, usa build_clean_quotes_run.ps1 con D:\quotes_clean_v2
  - no lances el nuevo Agent01 contra D:\quotes si quieres una separación limpia real

  Si quieres, el siguiente paso es que te cree una tercera lanzadera:

  - run_clean_quotes_agent01.ps1
    que encadene build_clean_quotes_run.ps1 + el comando final de descarga en un solo paso.